# Блокнот 5. kNN и SVM

Курс «Машинное обучение с нуля», модуль 5. Разделы: 1–2 — урок 5.1, 3 — уроки 5.2–5.3, 4 — урок 5.4, 5 — урок 5.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

rng = np.random.default_rng(0)

## 1. kNN своими руками (урок 5.1)

**Задание 1.** Допишите функцию: расстояния от `x` до всех обучающих точек, номера `k` ближайших (`np.argsort`), голосование (`np.bincount(...).argmax()`).

In [ ]:
def knn_predict(X_train, y_train, x, k):
    return ...  # ваш код

In [ ]:
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
pred = np.array([knn_predict(X_tr, y_tr, x, 5) for x in X_te[:100]])
sk = KNeighborsClassifier(5).fit(X_tr, y_tr).predict(X_te[:100])
assert (pred == sk).mean() > 0.97, "ответы должны почти совпадать с KNeighborsClassifier"
print("Верно! Доля верных ответов на 100 тестовых цифрах:", (pred == y_te[:100]).mean())

## 2. Масштабирование и проклятие размерности

На цифрах все признаки — яркости пикселей в одних и тех же единицах (0–16). Сравним kNN без масштабирования и со стандартизацией.

In [ ]:
for name, model in [("без масштабирования", KNeighborsClassifier(5)),
                    ("со стандартизацией", make_pipeline(StandardScaler(), KNeighborsClassifier(5)))]:
    print(name, cross_val_score(model, X_tr, y_tr, cv=5).mean().round(4))

Неожиданно: стандартизация здесь **вредит**. Краевые пиксели почти всегда равны нулю, а стандартизация растягивает их редкий шум до того же масштаба, что и информативные пиксели в центре. Масштабирование нужно, когда признаки в **разных** единицах; когда единицы общие, его стоит проверять, а не делать вслепую.

Проклятие размерности: отношение расстояния до ближайшего соседа к расстоянию до самого далёкого.

In [ ]:
for d in (2, 10, 100, 1000):
    pts = rng.uniform(size=(500, d))
    dist = np.linalg.norm(pts - rng.uniform(size=d), axis=1)
    print(f"d = {d:4d}: ближайший / самый далёкий = {dist.min() / dist.max():.3f}")

## 3. Линейный SVM, опорные векторы и hinge loss (уроки 5.2–5.3)

In [ ]:
A = np.r_[rng.normal([0, 0], 1, (40, 2)), rng.normal([3, 3], 1, (40, 2))]
b = np.r_[np.zeros(40), np.ones(40)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, C in zip(axes, (0.01, 100)):
    m = SVC(kernel="linear", C=C).fit(A, b)
    xx, yy = np.meshgrid(np.linspace(-3, 6, 200), np.linspace(-3, 6, 200))
    zz = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contour(xx, yy, zz, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="black")
    ax.scatter(*A.T, c=b, cmap="coolwarm", s=12)
    ax.scatter(*m.support_vectors_.T, s=80, facecolors="none", edgecolors="green")
    ax.set_title(f"C = {C}: опорных векторов {len(m.support_vectors_)}")
plt.show()

**Задание 2.** Напишите hinge loss: метки `y` из {−1, +1}, оценки модели `s`, потеря — среднее `max(0, 1 − y·s)` (`np.maximum`).

In [ ]:
def hinge_loss(y, s):
    return ...  # ваш код

In [ ]:
assert np.isclose(hinge_loss(np.array([1, -1, 1, -1]), np.array([2, -0.5, 0.3, 1])), 0.8)
print("Верно!")

## 4. Ядра: изогнутая граница (урок 5.4)

**Задание 3.** Посчитайте RBF-ядро двух точек: $K(x, z) = e^{-\gamma \|x - z\|^2}$.

In [ ]:
def rbf(x, z, gamma):
    return ...  # ваш код

In [ ]:
assert np.isclose(rbf(np.array([0, 0]), np.array([1, 1]), 1.0), np.exp(-2))
Xm, ym = make_moons(300, noise=0.25, random_state=0)
for name, m in [("линейное ядро", SVC(kernel="linear")), ("RBF, gamma = 1", SVC(kernel="rbf", gamma=1)),
                ("RBF, gamma = 50", SVC(kernel="rbf", gamma=50))]:
    print(f"{name:16s} accuracy по 5 фолдам: {cross_val_score(m, Xm, ym, cv=5).mean():.3f}")

## 5. Подбор C и gamma на цифрах (урок 5.5)

**Задание 4.** Запустите `GridSearchCV` по сетке: `C` из [0.1, 1, 10, 100], `gamma` из [1e-4, 1e-3, 1e-2, 1e-1]. В пайплайне шаг SVC называется `svc`, поэтому параметры пишутся `svc__C` и `svc__gamma`.

In [ ]:
pipe = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
grid = ...  # ваш код
print(grid.best_params_, round(grid.best_score_, 4))

In [ ]:
assert grid.best_score_ > 0.97
print("Верно! Тест:", round(grid.score(X_te, y_te), 4), "— опорных векторов:", grid.best_estimator_[-1].n_support_.sum(), "из", len(y_tr))
scores = grid.cv_results_["mean_test_score"].reshape(4, 4)
plt.imshow(scores, cmap="viridis")
plt.xticks(range(4), [1e-4, 1e-3, 1e-2, 1e-1])
plt.yticks(range(4), [0.1, 1, 10, 100])
plt.xlabel("gamma")
plt.ylabel("C")
plt.colorbar()
plt.show()